# Which detector should I build?

**Author:** Antoine ([@antowan](https://github.com/antowan))

**You can read every result here without running code or connecting a station.** All figures come from one Jetson Orin NX 16 GB (JetPack 6.0, TensorRT 8.6, FP16 engines), measured from 17 to 19 November 2026. They describe this device, not other devices.

**Accuracy and inference time** (sections 1 and 2):

| Model | mAP 50:95, 80 classes (published) | mAP 50:95, car / person / truck | FP16 ms per image, batch 1 | FP16 ms per image, batch 8 |
|---|---:|---:|---:|---:|
| YOLO26n | 39.92 (40.1) | 40.67 | 5.3 | 3.4 |
| YOLOv10n | 38.34 (38.5) | 40.54 | 5.5 | 3.8 |
| YOLOv7-tiny | 37.13 (n/a) | 39.41 | 8.4 | 4.7 |

**Live cameras per pipeline** (section 3): 30 FPS H.264 RTSP cameras that each kept at least 29 processed FPS.

| Model | 720p, no output | 720p, continuous recording + WebRTC + OSD | 720p, continuous recording + WebRTC + HLS + event recording + OSD | 1080p, no output |
|---|---:|---:|---:|---:|
| YOLO26n | 7 | 7 | 7 | 7 |
| YOLOv10n | 6† | 6 | 6 | 6 |
| YOLOv7-tiny | 4 | 4 | 4 | 4 |

† A higher count passed in one trial and failed in another; the table keeps the lower count.

For these three classes, YOLO26n and YOLOv10n are only **0.13 mAP points** apart. YOLOv7-tiny is **1.26 points** behind YOLO26n. YOLO26n also had the lowest raw inference time and handled the most cameras, which makes it a good [first build](03-build-yolo26.ipynb). Check difficult examples from your own cameras before choosing on accuracy alone.

## 1. How well does it detect?

A detector must name the object **and** put its box in the right place. Intersection over union (IoU) measures how much the predicted and labelled boxes overlap. **mAP 50:95** averages precision across classes and ten IoU thresholds, from 0.50 to 0.95. A score of 40 is not “40% of objects detected.”

Use **80 classes** to compare with published COCO results. On all 5,000 COCO val2017 images at 640 px, the raw FP16 engines scored within 0.2 points of the [YOLO26n end-to-end reference (40.1) and YOLOv10n paper reference (38.5)](../doc/benchmarks/orin-nx-16gb-jp6.0/comparison.md#accuracy). We found no directly comparable official 640 px result for these YOLOv7-tiny weights. The **three-class** column scores the same predictions for car, person and truck only; it is not a published benchmark or a test of the deployed callback.

### Accuracy by class

Scores of the TensorRT FP16 engine on all 5,000 COCO val2017 images, in percent. **AP 50** accepts a box from 50% overlap, a looser test than mAP 50:95. Read across a row to see whether a similar overall score hides a difference on a class you care about. These evaluations use labelled images rather than tracked video.

| Model | Published COCO mAP 50:95 | Measured mAP 50:95 (80 classes) | AP 50 | mAP 50:95 (car, person, truck) | AP car | AP person | AP truck |
|---|---:|---:|---:|---:|---:|---:|---:|
| YOLO26n | 40.1 | 39.92 | 55.57 | 40.67 | 37.02 | 52.69 | 32.32 |
| YOLOv10n | 38.5 | 38.34 | 53.55 | 40.54 | 37.17 | 52.46 | 31.98 |
| YOLOv7-tiny | n/a | 37.13 | 54.64 | 39.41 | 36.65 | 50.50 | 31.09 |

## 2. How long does inference take?

The first table divides the median batch time by the number of images in the batch. In brackets: images per second and the speed-up over ONNX Runtime CUDA FP32 (ORT). The second table gives the time for the **whole request**. A batch of eight is not eight cameras running live.

Timing used GPU-resident synthetic inputs and excludes decoding, tracking and video outputs. For YOLOv7-tiny it also excludes the deployment engine’s EfficientNMS plugin. The idle timing run used rebuilt engines; the accuracy results are from an earlier run. See the [measurement conditions](../doc/benchmarks/orin-nx-16gb-jp6.0/conditions.md).

| Model | Batch 1 (ms per image) | Batch 4 (ms per image) | Batch 8 (ms per image) |
|---|---:|---:|---:|
| YOLO26n | 5.31 (188 img/s, 3.6x ORT) | 3.76 (266 img/s, 3.8x ORT) | 3.42 (292 img/s, 3.9x ORT) |
| YOLOv10n | 5.50 (182 img/s, 3.3x ORT) | 4.09 (244 img/s, 3.7x ORT) | 3.76 (266 img/s, 3.7x ORT) |
| YOLOv7-tiny | 8.40 (119 img/s, 2.9x ORT) | 5.31 (188 img/s, 4.3x ORT) | 4.74 (211 img/s, 4.1x ORT) |

**All runtimes and precisions** (median time of one request):

| Model | Batch | ONNX Runtime CUDA FP32 | TensorRT FP32 | TensorRT FP16 |
|---|---:|---:|---:|---:|
| YOLO26n | 1 | 19.0 ms | 12.2 ms | 5.3 ms |
| YOLO26n | 4 | 56.9 ms | 38.5 ms | 15.0 ms |
| YOLO26n | 8 | 106.1 ms | 72.0 ms | 27.4 ms |
| YOLOv10n | 1 | 18.4 ms | 12.7 ms | 5.5 ms |
| YOLOv10n | 4 | 61.4 ms | 40.3 ms | 16.4 ms |
| YOLOv10n | 8 | 112.2 ms | 76.5 ms | 30.1 ms |
| YOLOv7-tiny | 1 | 24.4 ms | 18.0 ms | 8.4 ms |
| YOLOv7-tiny | 4 | 90.8 ms | 64.5 ms | 21.3 ms |
| YOLOv7-tiny | 8 | 154.7 ms | 124.4 ms | 37.9 ms |

### See the trade-off

Lower latency is better (left); higher mAP is better (right).

![Left: TensorRT FP16 milliseconds per image at batch 1, 4 and 8 for YOLO26n, YOLOv10n and YOLOv7-tiny. Right: measured COCO mAP 50:95 on 80 classes next to the published figure.](../doc/benchmarks/orin-nx-16gb-jp6.0/comparison.png)

### Does FP16 change detection quality?

Compare FP32 and FP16 scores at the **same batch size**. The difference is in mAP points, not percent speed-up.

| Model | Batch | mAP 50:95 TensorRT FP32 | mAP 50:95 TensorRT FP16 | Difference (points) |
|---|---:|---:|---:|---:|
| YOLO26n | 8 | 39.93 | 39.92 | -0.00 |
| YOLOv10n | 8 | 38.33 | 38.34 | +0.01 |
| YOLOv7-tiny | 8 | 37.15 | 37.13 | -0.02 |

## 3. How many live cameras does that mean?

Don’t divide model throughput by 30. A camera frame must also be decoded, tracked and, if requested, encoded for output. The camera counts at the top come from live tests with **30 FPS H.264 RTSP cameras** and one pipeline at a time. A count passes when every camera averages at least **29 processed FPS** over 120 seconds, after a 45-second warm-up; that is near real time, not proof that every frame was processed. Each reported count passed at least twice, and the next count failed.

The output columns use a 1280x720 mosaic with on-screen display (OSD) of the detections. WebRTC and HLS are delivered at 2 Mbps, continuous recording at 3 Mbps and event recording at 4 Mbps.

### With and without video outputs

All three output profiles reached the same counts, but outputs use part of the margin. This table lists the slowest camera of every trial at the first count that failed, in FPS (29 needed):

| Model | Cameras | No output | Continuous recording + WebRTC + OSD | Continuous recording + WebRTC + HLS + event recording + OSD |
|---|---:|---|---|---|
| YOLO26n | 8 | 28.20, 28.10 | 27.33 | 26.57 |
| YOLOv10n | 7 | 29.64 (passed), 28.91 | 28.39 | 27.50 |
| YOLOv7-tiny | 5 | 24.58, 24.38 | 25.64 | 25.31 |

For example, YOLO26n at eight cameras dropped from about 28.1 FPS with no output to 26.6 FPS with all outputs. These trials did not run at the same time or temperature, so they don’t isolate an exact output cost.

### Other output settings (YOLO26n)

Each row is one YOLO26n run at seven 720p cameras, changing one setting at a time. All passed. WebRTC means the encoder and publisher with **no connected viewer**. Continuous recordings never carry OSD.

| Model | Cameras | Outputs | Slowest camera FPS | Mean GPU busy |
|---|---:|---|---:|---:|
| YOLO26n | 7 | No output | 30.51 | 91.7% |
| YOLO26n | 7 | Continuous recording | 30.35 | 96.6% |
| YOLO26n | 7 | WebRTC | 30.45 | 94.6% |
| YOLO26n | 7 | WebRTC + OSD | 30.23 | 95.0% |
| YOLO26n | 7 | HLS | 30.34 | 95.1% |
| YOLO26n | 7 | HLS + OSD | 30.21 | 96.6% |
| YOLO26n | 7 | Event recording | 30.29 | 95.9% |
| YOLO26n | 7 | Event recording + OSD | 30.20 | 94.8% |
| YOLO26n | 7 | Continuous recording + WebRTC + OSD | 30.13 | 96.2% |
| YOLO26n | 7 | WebRTC + OSD, 1 Mbps | 30.17 | 96.3% |
| YOLO26n | 7 | WebRTC + OSD, 4 Mbps | 30.19 | 95.2% |
| YOLO26n | 7 | WebRTC + OSD, 640x360 mosaic | 30.09 | 95.9% |
| YOLO26n | 7 | WebRTC + OSD, 1920x1080 mosaic | 30.10 | 96.1% |

### What was not tested

- Connected WebRTC viewers or HLS playback in a browser. Viewers add load.
- Video outputs with 1080p cameras.
- Output settings outside the camera table for YOLOv10n and YOLOv7-tiny.
- Several pipelines sharing the station.
- Long runs and cooling. The GPU junction reached 88.3 °C during these tests.
- Busy scenes. Every camera looped the same street clip.

### Known issue: one camera with WebRTC or recording

On this development build, **YOLO26n with one camera failed with WebRTC + OSD in all 11 attempts and with continuous recording in both attempts**. MediaMTX reported hardware-encoder errors, although seven cameras passed. HLS + OSD with one camera passed. A maximum of seven doesn’t prove that every count from one to seven works.

The [capacity report](../doc/benchmarks/orin-nx-16gb-jp6.0/capacity.md) has every repeat, the 1080p evidence and the output diagnostics.

## 4. When was this measured?

All tests used the same Orin NX 16 GB with JetPack 6.0 (L4T R36.3) and TensorRT 8.6; the timing and camera tests ran in MAXN power mode. Dates are UTC.

| Measurement | Date | What ran | Evidence |
|---|---|---|---|
| Accuracy | 17 November 2026 | 5,000 COCO val2017 images per model: ONNX Runtime FP32, TensorRT FP32 and FP16 | [YOLO26n](../doc/benchmarks/orin-nx-16gb-jp6.0/yolo26-accuracy.json), [YOLOv10n](../doc/benchmarks/orin-nx-16gb-jp6.0/yolov10-accuracy.json), [YOLOv7-tiny](../doc/benchmarks/orin-nx-16gb-jp6.0/yolov7-accuracy.json) |
| Inference time | 18 November 2026 | Raw engines on an idle GPU, with the video pipelines stopped | [YOLO26n](../doc/benchmarks/orin-nx-16gb-jp6.0/yolo26-latency.json), [YOLOv10n](../doc/benchmarks/orin-nx-16gb-jp6.0/yolov10-latency.json), [YOLOv7-tiny](../doc/benchmarks/orin-nx-16gb-jp6.0/yolov7-latency.json), [idle checks](../doc/benchmarks/orin-nx-16gb-jp6.0/idle-conditions.json) |
| Camera capacity | 18 to 19 November 2026 | 80 live RTSP trials with DeepStream 7.0 | [`capacity.jsonl`](../doc/benchmarks/orin-nx-16gb-jp6.0/capacity.jsonl) |
| Output checks | 19 November 2026 | 21 YOLO26n output diagnostics | [`capacity-outputs.json`](../doc/benchmarks/orin-nx-16gb-jp6.0/capacity-outputs.json) |

The [measurement conditions](../doc/benchmarks/orin-nx-16gb-jp6.0/conditions.md) describe the station state during each run.

## 5. Repeat the model measurements (optional, station only)

Leave `MEASURE = False` to skip **all** downloads, installs and benchmarks below. To make new measurements, use App-Builder on an idle development station with internet access and room for COCO val2017 and the engines. Reserve the GPU rather than stopping someone else’s pipeline. Allow roughly 40–60 minutes per flavor plus the download.

The cells export models, build raw engines, time inference and score labelled images. Results go to `tmp/benchmarks`, not the reference directory. Use `LIMIT = 0` for a full COCO comparison; a subset is only a smoke test. These commands do not run the live camera-capacity tests.

In [ ]:
import os
from pathlib import Path

if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)
assert Path("tools/compare_models.py").is_file(), "Open this notebook from the repository (notebooks/ folder)"

MEASURE = False          # set to True to run the cells below
LIMIT = 0                # score only the first LIMIT images (0 = all 5,000); use e.g. 200 for a quick check
FLAVORS = ["yolov7", "yolov10", "yolo26"]

os.environ.update(MEASURE=str(int(MEASURE)), LIMIT=str(LIMIT), FLAVORS=" ".join(FLAVORS))

In [ ]:
%%bash
set -euo pipefail
[ "$MEASURE" = 1 ] || { echo "MEASURE is False: skipped"; exit 0; }
mkdir -p tmp/coco && cd tmp/coco
[ -d val2017 ] || { curl -fsSLO http://images.cocodataset.org/zips/val2017.zip && unzip -q val2017.zip && rm val2017.zip; }
[ -f annotations/instances_val2017.json ] || { curl -fsSLO http://images.cocodataset.org/annotations/annotations_trainval2017.zip \
    && unzip -q -o annotations_trainval2017.zip annotations/instances_val2017.json && rm annotations_trainval2017.zip; }
python3 -m pip install --user -r ../../tools/requirements-accuracy.txt
python3 -m pip install --user --no-deps -r ../../tools/requirements-export.txt
ls val2017 | wc -l

In [ ]:
%%bash
set -euo pipefail
[ "$MEASURE" = 1 ] || { echo "MEASURE is False: skipped"; exit 0; }
for flavor in $FLAVORS; do
    python3 tools/export_model.py "$flavor"
done

In [ ]:
%%bash
set -euo pipefail
[ "$MEASURE" = 1 ] || { echo "MEASURE is False: skipped"; exit 0; }
for flavor in $FLAVORS; do
    python3 tools/measure.py "$flavor" --annotations tmp/coco/annotations/instances_val2017.json \
        --images tmp/coco/val2017 --limit "$LIMIT" --results tmp/benchmarks
done

In [ ]:
%%bash
set -euo pipefail
[ "$MEASURE" = 1 ] || { echo "MEASURE is False: skipped"; exit 0; }
python3 tools/compare_models.py tmp/benchmarks

The next cell shows your report, `tmp/benchmarks/comparison.md`, and its chart. The tables above keep describing the reference runs. Live camera capacity is a separate test; see the [capacity test plan](../doc/benchmarks/capacity/README.md).

In [ ]:
from IPython.display import Image, Markdown, display

OWN = Path("tmp/benchmarks")
if (OWN / "comparison.md").is_file():
    display(Markdown((OWN / "comparison.md").read_text()))
    if (OWN / "comparison.png").is_file():
        display(Image(filename=str(OWN / "comparison.png")))
else:
    print("No report in tmp/benchmarks yet: set MEASURE = True and run the cells above.")